# IM3 × EIA RTO join

Adds grid-level (balancing authority) features to the cleaned IM3 base, using two `electricity/rto` sub-routes:



In [18]:
# 1. Setup
import os, glob, time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import requests
from sklearn.neighbors import BallTree
from dotenv import load_dotenv

load_dotenv()
KEY = os.getenv("EIA_API_KEY")   # <-- use the same variable name your .env uses
assert KEY, "No EIA key found. Check the variable name in your .env"

DATA = Path("Data")
YEAR = 2025        # last full calendar year
K = 5              # nearest plants that vote on each data center's BA
MAX_KM = 100       # if the nearest plant is farther than this, leave the BA blank
EARTH_KM = 6371.0

In [19]:
# 2. EIA API helpers: paging, automatic retries, and one-BA-at-a-time downloads
RETRY_CODES = {429, 500, 502, 503, 504}   # EIA busy / timed out -> wait and try again


def get_with_retry(url, params, tries=6):
    for attempt in range(1, tries + 1):
        try:
            r = requests.get(url, params=params, timeout=120)
            if r.status_code not in RETRY_CODES:
                r.raise_for_status()          # real errors (bad key, bad facet) stop right away
                return r.json()["response"]
            err = f"HTTP {r.status_code}"
        except (requests.ConnectionError, requests.Timeout) as e:
            err = type(e).__name__
        if attempt == tries:
            raise RuntimeError(f"EIA kept failing ({err}) after {tries} tries. Wait a few minutes and rerun this cell.")
        wait = 10 * attempt
        print(f"   {err}, retrying in {wait}s (try {attempt + 1}/{tries})")
        time.sleep(wait)


def eia_get(route, params):
    url = f"https://api.eia.gov/v2/{route}/data/"
    rows, offset = [], 0
    while True:
        resp = get_with_retry(url, {**params, "api_key": KEY, "offset": offset, "length": 5000})
        rows.extend(resp["data"])
        total = int(resp["total"])
        offset += 5000
        if offset >= total:
            break
        time.sleep(0.2)
    return pd.DataFrame(rows)


def eia_get_by_ba(route, params, bas, cache_dir):
    """Download one BA at a time and save each to its own CSV.
    If anything fails, rerun the cell: finished BAs load from disk and it picks up where it stopped."""
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    parts = []
    for i, ba in enumerate(bas, 1):
        f = cache_dir / f"{ba}.csv"
        if f.exists():
            parts.append(pd.read_csv(f, dtype={"respondent": str}))
            continue
        df = eia_get(route, {**params, "facets[respondent][]": ba})
        if df.empty:   # BA reports nothing for this route; save an empty file so it isn't re-asked
            df = pd.DataFrame(columns=["period", "respondent", "value"])
        df.to_csv(f, index=False)
        parts.append(df)
        print(f"[{i}/{len(bas)}] {ba}: {len(df):,} rows")
    return pd.concat(parts, ignore_index=True)


def facet_ids(route, facet):
    url = f"https://api.eia.gov/v2/{route}/facet/{facet}/"
    return [f["id"] for f in get_with_retry(url, {"api_key": KEY})["facets"]]

In [20]:
# 3. Load the cleaned IM3 base, with the exact same cleaning as the generator join
im3_csvs = [p for p in glob.glob("Data/*.csv") if Path(p).name.lower().startswith("im3")]
assert len(im3_csvs) == 1, f"expected exactly one IM3 CSV in Data/, found {im3_csvs}"

ID_DTYPES = {"id": str, "state_id": str, "county_id": str}   # keep leading zeros
base = pd.read_csv(im3_csvs[0], dtype=ID_DTYPES)
base = base.drop_duplicates()
base = base.drop_duplicates(subset=["id"]).reset_index(drop=True)
base = base[base["sqft"] > 0]   # drops the 'point' rows (no sqft), same as before
base = base[base["lat"].between(17.0, 72.0) & base["lon"].between(-170.0, -65.0)].reset_index(drop=True)

N_BASE = len(base)
assert base["id"].is_unique
print(N_BASE, "facilities (should match the generator join: 1147)")

1147 facilities (should match the generator join: 1147)


In [21]:
# 4. Load the raw generator table saved by the generator join, and find its BA column
gen_files = sorted(glob.glob("Data/eia_generators_*_raw.csv"))
assert gen_files, "No raw generator CSV in Data/. Run im3_eia_join.ipynb first."
gens = pd.read_csv(gen_files[-1], dtype=str)
print("Using", gen_files[-1], f"({len(gens):,} rows)")

ba_cols = [c for c in gens.columns if "balancing" in c.lower() and "code" in c.lower()]
assert ba_cols, f"No balancing authority code column found. Columns are: {list(gens.columns)}"
BA_COL = ba_cols[0]
PLANT_COL = "plantid"
print("BA column:", BA_COL)

Using Data/eia_generators_2026-07_raw.csv (26,072 rows)
BA column: balancing_authority_code


In [22]:
# 5. One row per plant, keeping only plants whose BA reports to the RTO data
rto_codes = set(facet_ids("electricity/rto/region-data", "respondent"))

plants = gens[[PLANT_COL, "latitude", "longitude", BA_COL]].copy()
plants["latitude"] = pd.to_numeric(plants["latitude"], errors="coerce")
plants["longitude"] = pd.to_numeric(plants["longitude"], errors="coerce")
plants["ba"] = plants[BA_COL].astype(str).str.strip().str.upper()
plants = plants.dropna(subset=["latitude", "longitude"])

dropped = plants.loc[~plants["ba"].isin(rto_codes), "ba"].value_counts()
print("Plant BA codes not in RTO data (dropped):", dropped.to_dict())
plants = plants[plants["ba"].isin(rto_codes)]

plants = (plants.groupby(PLANT_COL, as_index=False)      # one row per plant, not per generator
                .agg(latitude=("latitude", "first"),
                     longitude=("longitude", "first"),
                     ba=("ba", "first")))
print(f"{len(plants):,} plants with a valid BA code")

Plant BA codes not in RTO data (dropped): {'HECO': 34, 'NBSO': 14, 'GRIS': 2, 'CEA': 1}
13,922 plants with a valid BA code


In [23]:
# 6. Give each data center a BA code: vote among its nearest plants in the SAME county
#    (matched by county FIPS), falling back to the same state, then to any plant.
#    Plants get their county from Census county boundaries (one ~10 MB download, saved to Data/).
#    Two corrections on top of the plant vote:
#      - generation-only BAs (wind/solar/gas owners with no customers) can't be a data center's grid
#      - a few counties where the local utility runs its own BA but its plants are registered elsewhere
import geopandas as gpd

GEN_ONLY_BAS = {"AVRN", "DEAA", "EEI", "GRID", "GRIF", "GWA", "HGMA", "SEPA", "WWA", "YAD"}

# county FIPS -> BA of the utility that actually serves the county (EIA-861 utility territories)
COUNTY_OVERRIDES = {
    "53025": ("GCPD", "Grant County WA: Grant County PUD (Quincy)"),
    "53017": ("DOPD", "Douglas County WA: Douglas County PUD (East Wenatchee)"),
    "41059": ("BPAT", "Umatilla County OR: Umatilla Electric Co-op / City of Hermiston, BPA customers"),
    "41065": ("BPAT", "Wasco County OR: Northern Wasco County PUD (The Dalles), BPA customer"),
}
for fips, (ba, why) in COUNTY_OVERRIDES.items():
    assert ba in rto_codes, f"{ba} is not an RTO respondent; remove the {fips} override"

K = 5             # plants that vote
K_SEARCH = 300    # how many nearest plants to look through for same-county / same-state ones

county_zip = DATA / "cb_2023_us_county_500k.zip"
if not county_zip.exists():
    r = requests.get("https://www2.census.gov/geo/tiger/GENZ2023/shp/cb_2023_us_county_500k.zip", timeout=180)
    r.raise_for_status()
    county_zip.write_bytes(r.content)
counties = gpd.read_file(county_zip)[["GEOID", "STATEFP", "geometry"]].to_crs("EPSG:4326")

# county FIPS + state FIPS for every plant (rerun-safe)
plants = plants.drop(columns=["p_fips", "p_state"], errors="ignore")
pts = gpd.GeoDataFrame(plants[[PLANT_COL]],
                       geometry=gpd.points_from_xy(plants["longitude"], plants["latitude"]),
                       crs="EPSG:4326")
pts = gpd.sjoin(pts, counties, how="left", predicate="within").drop_duplicates(PLANT_COL)
plants = plants.merge(pts[[PLANT_COL, "GEOID", "STATEFP"]]
                        .rename(columns={"GEOID": "p_fips", "STATEFP": "p_state"}),
                      on=PLANT_COL, how="left")
print(f"{plants['p_fips'].notna().mean():.1%} of plants placed in a county "
      "(the rest are offshore/on the coastline and use the fallback)")

n_before = len(plants)
plants = plants[~plants["ba"].isin(GEN_ONLY_BAS)].reset_index(drop=True)
print(n_before - len(plants), "plants dropped because their BA is generation-only")

tree = BallTree(np.radians(plants[["latitude", "longitude"]].values), metric="haversine")
dist, idx = tree.query(np.radians(base[["lat", "lon"]].astype(float).values),
                       k=min(K_SEARCH, len(plants)))   # sorted closest first

p_ba = plants["ba"].values
p_st = plants["p_state"].fillna("").values
p_co = plants["p_fips"].fillna("").values
f_st = base["state_id"].str.zfill(2).values
f_co = (base["state_id"].str.zfill(2) + base["county_id"].str.zfill(3)).values

ba_codes, shares, levels = [], [], []
for i in range(len(base)):
    cand = idx[i]
    same_county = cand[p_co[cand] == f_co[i]][:K]
    same_state = cand[p_st[cand] == f_st[i]][:K]
    if len(same_county):
        pick, level = same_county, "county"
    elif len(same_state):
        pick, level = same_state, "state"
    else:
        pick, level = cand[:K], "nearest"
    ba, n = Counter(p_ba[pick]).most_common(1)[0]   # ties go to the closer plant
    ba_codes.append(ba)
    shares.append(n / len(pick))
    levels.append(level)

rto = base[["id", "state_abb", "county", "lat", "lon"]].copy()
rto["ba_code"] = ba_codes
rto["vote_share"] = shares
rto["matched_by"] = levels
rto["nearest_plant_km"] = dist[:, 0] * EARTH_KM

fac_fips = pd.Series(f_co, index=rto.index)
for fips, (ba, why) in COUNTY_OVERRIDES.items():
    hit = fac_fips == fips
    if hit.any():
        print(f"Override: {hit.sum():>3} facilities -> {ba}  ({why})")
    rto.loc[hit, "ba_code"] = ba
    rto.loc[hit, "matched_by"] = "override"

too_far = rto["nearest_plant_km"] > MAX_KM
rto.loc[too_far, "ba_code"] = np.nan
print("How each facility was matched:", rto["matched_by"].value_counts().to_dict())
print(too_far.sum(), "facilities farther than", MAX_KM, "km from any plant (BA left blank)")
print()
print("WA / OR check (county -> BA):")
print(rto[rto["state_abb"].isin(["WA", "OR"])]
        .groupby(["state_abb", "county"])["ba_code"]
        .agg(lambda s: s.value_counts().to_dict())
        .to_string())

99.9% of plants placed in a county (the rest are offshore/on the coastline and use the fallback)
39 plants dropped because their BA is generation-only
Override:  38 facilities -> GCPD  (Grant County WA: Grant County PUD (Quincy))
Override:  13 facilities -> DOPD  (Douglas County WA: Douglas County PUD (East Wenatchee))
Override:  25 facilities -> BPAT  (Umatilla County OR: Umatilla Electric Co-op / City of Hermiston, BPA customers)
Override:   2 facilities -> BPAT  (Wasco County OR: Northern Wasco County PUD (The Dalles), BPA customer)
How each facility was matched: {'county': 1051, 'override': 78, 'state': 18}
0 facilities farther than 100 km from any plant (BA left blank)

WA / OR check (county -> BA):
state_abb  county           
OR         Clackamas County       {'PGE': 1}
           Crook County         {'PACW': 13}
           Deschutes County      {'PACW': 1}
           Marion County          {'PGE': 2}
           Morrow County        {'BPAT': 23}
           Multnomah County     

In [24]:
# 7. Sanity check: BA by state. Expect e.g. TX -> ERCO, VA -> PJM, CA -> CISO/BANC/LDWP/IID
top_states = rto["state_abb"].value_counts().head(10).index
print(pd.crosstab(rto.loc[rto["state_abb"].isin(top_states), "state_abb"],
                  rto.loc[rto["state_abb"].isin(top_states), "ba_code"])
        .apply(lambda r: r[r > 0].sort_values(ascending=False).to_dict(), axis=1)
        .to_string())

state_abb
AZ                              {'AZPS': 20, 'SRP': 20}
CA                   {'CISO': 72, 'BANC': 6, 'LDWP': 5}
IA                                         {'MISO': 56}
IL                               {'PJM': 35, 'MISO': 1}
NJ                                          {'PJM': 45}
OH                                          {'PJM': 55}
OR                  {'BPAT': 50, 'PGE': 27, 'PACW': 14}
TX                                         {'ERCO': 89}
VA                                         {'PJM': 288}
WA    {'GCPD': 38, 'PSEI': 19, 'DOPD': 13, 'AVA': 4,...


In [25]:
# 8. Download one year of hourly demand, one BA at a time (saved per BA, so reruns are instant)
bas = sorted(rto["ba_code"].dropna().unique())
print(len(bas), "BAs:", bas)

demand = eia_get_by_ba("electricity/rto/region-data", {
    "frequency": "hourly",
    "data[]": "value",
    "facets[type][]": "D",                 # D = demand
    "start": f"{YEAR}-01-01T00",
    "end": f"{YEAR}-12-31T23",
    "sort[0][column]": "period", "sort[0][direction]": "asc",
}, bas, DATA / f"eia_rto_demand_{YEAR}")
print(f"{len(demand):,} hourly rows")

39 BAs: ['AVA', 'AZPS', 'BANC', 'BPAT', 'CHPD', 'CISO', 'CPLE', 'DOPD', 'DUK', 'EPE', 'ERCO', 'FMPP', 'FPC', 'FPL', 'GCPD', 'GVL', 'ISNE', 'JEA', 'LDWP', 'LGEE', 'MISO', 'NEVP', 'NWMT', 'NYIS', 'PACE', 'PACW', 'PGE', 'PJM', 'PNM', 'PSCO', 'PSEI', 'SC', 'SCEG', 'SOCO', 'SRP', 'SWPP', 'TEC', 'TVA', 'WACM']
[8/39] DOPD: 8,760 rows
   HTTP 504, retrying in 10s (try 2/6)
[12/39] FMPP: 8,760 rows
[15/39] GCPD: 8,760 rows
341,613 hourly rows


In [26]:
# 9. Clean demand and aggregate to one row per BA
d = demand[["period", "respondent", "value"]].copy()
d["value"] = pd.to_numeric(d["value"], errors="coerce")
d = d.drop_duplicates(subset=["period", "respondent"])
d = d[d["value"] > 0]                                  # drops missing hours and bad negative readings

# EIA-930 hourly data has occasional glitch spikes; drop readings over 4x the BA's median
med = d.groupby("respondent")["value"].transform("median")
spikes = d["value"] > 4 * med
print(spikes.sum(), "spike readings dropped")
d = d[~spikes]

HOURS = 8784 if pd.Timestamp(f"{YEAR}-12-31").is_leap_year else 8760
ba_demand = (d.groupby("respondent")["value"]
              .agg(ba_peak_mw="max", ba_avg_mw="mean", hours="count")
              .reset_index()
              .rename(columns={"respondent": "ba_code"}))
ba_demand["ba_peak_to_avg"] = ba_demand["ba_peak_mw"] / ba_demand["ba_avg_mw"]
ba_demand["ba_demand_coverage"] = ba_demand["hours"] / HOURS
ba_demand = ba_demand.drop(columns="hours")

print("BAs with under 90% of hours reported:")
print(ba_demand.loc[ba_demand["ba_demand_coverage"] < 0.9, ["ba_code", "ba_demand_coverage"]])
print("BAs with no demand data (generation-only BAs, these get NaN):",
      sorted(set(bas) - set(ba_demand["ba_code"])))
ba_demand.sort_values("ba_peak_mw", ascending=False).head(10)

0 spike readings dropped
BAs with under 90% of hours reported:
Empty DataFrame
Columns: [ba_code, ba_demand_coverage]
Index: []
BAs with no demand data (generation-only BAs, these get NaN): []


,ba_code,ba_peak_mw,ba_avg_mw,ba_peak_to_avg,ba_demand_coverage
27,PJM,160560,96241.618265,1.668301,1.000000
20,MISO,118661,75772.987443,1.566007,1.000000
10,ERCO,83597,55711.806393,1.500526,1.000000
35,SWPP,54745,34226.006050,1.599515,1.000000
33,SOCO,46490,27342.227626,1.700300,1.000000
5,CISO,43860,25573.310388,1.715069,1.000000
37,TVA,35086,19488.310274,1.800361,1.000000
23,NYIS,31857,17302.089176,1.841223,0.999772
13,FPL,28963,16730.491553,1.731151,1.000000
16,ISNE,25898,13158.212785,1.968200,1.000000


In [27]:
# 10. Download daily generation by fuel type (one timezone only, or rows get duplicated)
tzs = facet_ids("electricity/rto/daily-fuel-type-data", "timezone")
TZ = "Central" if "Central" in tzs else tzs[0]
print("Valid timezones:", tzs, "-> using", TZ)

fuel = eia_get_by_ba("electricity/rto/daily-fuel-type-data", {
    "frequency": "daily",
    "data[]": "value",
    "facets[timezone][]": TZ,
    "start": f"{YEAR}-01-01",
    "end": f"{YEAR}-12-31",
    "sort[0][column]": "period", "sort[0][direction]": "asc",
}, bas, DATA / f"eia_rto_fuel_{YEAR}")
print(f"{len(fuel):,} daily rows")

Valid timezones: ['Central', 'Pacific', 'Mountain', 'Arizona', 'Eastern'] -> using Central
[8/39] DOPD: 365 rows
[12/39] FMPP: 1,902 rows
[15/39] GCPD: 365 rows
88,038 daily rows


In [28]:
# 11. Clean fuel data and turn it into fuel-mix shares per BA
f = fuel.reindex(columns=["period", "respondent", "fueltype", "value"]).dropna(subset=["respondent", "fueltype"]).copy()
f["value"] = pd.to_numeric(f["value"], errors="coerce")
f = f.drop_duplicates(subset=["period", "respondent", "fueltype"])
f = f[f["value"] > 0]      # negatives are storage charging / net consumption, not generation
print("Fuel codes present:", sorted(f["fueltype"].unique()))

annual = f.pivot_table(index="respondent", columns="fueltype", values="value",
                       aggfunc="sum", fill_value=0)

# Storage (batteries, pumped hydro) only re-releases power made earlier, so it isn't a fuel source
STORAGE = ["BAT", "PS", "OES", "UES"]
annual = annual.drop(columns=[c for c in STORAGE if c in annual.columns])
total = annual.sum(axis=1)

def share(codes):
    return annual[[c for c in codes if c in annual.columns]].sum(axis=1) / total

SOLAR = ["SUN", "SNB"]          # SNB = solar with an attached battery (still solar generation)
WIND = ["WND", "WNB"]           # WNB = wind with an attached battery
FOSSIL = ["COL", "NG", "OIL"]
RENEWABLE = SOLAR + WIND + ["WAT", "GEO"]   # same definition as the generator join (no nuclear)
NUCLEAR = ["NUC"]

ba_fuel = pd.DataFrame({
    "ba_share_coal":      share(["COL"]),
    "ba_share_gas":       share(["NG"]),
    "ba_share_nuclear":   share(NUCLEAR),
    "ba_share_hydro":     share(["WAT"]),
    "ba_share_solar":     share(SOLAR),
    "ba_share_wind":      share(WIND),
    "ba_share_fossil":    share(FOSSIL),
    "ba_share_renewable": share(RENEWABLE),
    "ba_share_clean":     share(RENEWABLE + NUCLEAR),   # renewable + nuclear (carbon-free)
    "ba_total_gen_mwh":   total,
}).reset_index().rename(columns={"respondent": "ba_code"})

assert (ba_fuel.filter(like="ba_share").fillna(0) <= 1.0001).all().all()
ba_fuel.sort_values("ba_share_solar", ascending=False).head(10)

Fuel codes present: ['BAT', 'COL', 'GEO', 'NG', 'NUC', 'OES', 'OIL', 'OTH', 'PS', 'SNB', 'SUN', 'WAT', 'WND']


,ba_code,ba_share_coal,ba_share_gas,ba_share_nuclear,ba_share_hydro,ba_share_solar,ba_share_wind,ba_share_fossil,ba_share_renewable,ba_share_clean,ba_total_gen_mwh
28,PNM,0.115777,0.165053,0.000000,0.002204,0.303088,0.410321,0.280830,0.719060,0.719060,15152061
5,CISO,0.000004,0.419457,0.093452,0.113370,0.264257,0.105580,0.421350,0.484623,0.578075,188215486
21,NEVP,0.069932,0.559965,0.000000,0.007234,0.243149,0.009460,0.629898,0.259843,0.259843,36851957
18,LDWP,0.164107,0.399923,0.000000,0.046939,0.219959,0.101586,0.564031,0.368485,0.368485,18206450
1,AZPS,0.274692,0.422400,0.000000,0.000000,0.212108,0.072086,0.697091,0.284194,0.284194,23748188
25,PACW,0.000000,0.049420,0.000000,0.395985,0.178940,0.307257,0.049420,0.882183,0.882183,5514896
9,EPE,0.000000,0.842138,0.000000,0.000000,0.157862,0.000000,0.842138,0.157862,0.157862,6802599
10,ERCO,0.129541,0.408587,0.085863,0.001101,0.138478,0.235868,0.538128,0.375447,0.461310,486616150
36,TEC,0.000002,0.864224,0.000000,0.000000,0.119338,0.000000,0.864419,0.119338,0.119338,18404296
24,PACE,0.441869,0.177452,0.000000,0.019819,0.115281,0.227766,0.619321,0.362866,0.362866,60827434


In [ ]:
# 12. Join everything onto the IM3 base (one row per facility)
rto_features = (rto[["id", "ba_code"]]
                .merge(ba_demand, on="ba_code", how="left")
                .merge(ba_fuel, on="ba_code", how="left"))

assert len(rto_features) == N_BASE, "row count changed during the join"
assert rto_features["id"].is_unique, "duplicate ids after the join"

print("Share of missing values per column:")
print(rto_features.isna().mean().round(3).sort_values(ascending=False))

rto_features.to_csv(DATA / "eia_rto_features.csv", index=False)
print("Saved", DATA / "eia_rto_features.csv", rto_features.shape)
rto_features.head()

Share of missing values per column:
id                    0.0
ba_code               0.0
ba_peak_mw            0.0
ba_avg_mw             0.0
ba_peak_to_avg        0.0
ba_demand_coverage    0.0
ba_share_coal         0.0
ba_share_gas          0.0
ba_share_nuclear      0.0
ba_share_hydro        0.0
ba_share_solar        0.0
ba_share_wind         0.0
ba_share_fossil       0.0
ba_share_renewable    0.0
ba_share_clean        0.0
ba_total_gen_mwh      0.0
dtype: float64
Saved Data/eia_rto_features.csv (1147, 16)


,id,ba_code,ba_peak_mw,ba_avg_mw,ba_peak_to_avg,ba_demand_coverage,ba_share_coal,ba_share_gas,ba_share_nuclear,ba_share_hydro,ba_share_solar,ba_share_wind,ba_share_fossil,ba_share_renewable,ba_share_clean,ba_total_gen_mwh
0,0002744301,PJM,160560,96241.618265,1.668301,1.000000,0.166368,0.422042,0.308040,0.018137,0.029995,0.035004,0.594643,0.083136,0.391175,765599358
1,0007805491,PJM,160560,96241.618265,1.668301,1.000000,0.166368,0.422042,0.308040,0.018137,0.029995,0.035004,0.594643,0.083136,0.391175,765599358
2,0009474864,DUK,21931,12535.759219,1.749475,0.999886,0.187517,0.242843,0.538021,0.013686,0.017383,0.000000,0.430360,0.031069,0.569090,100070105
3,0013924557,MISO,118661,75772.987443,1.566007,1.000000,0.290617,0.355727,0.138180,0.015676,0.047850,0.146286,0.646345,0.209812,0.347992,576768527
4,0014593270,DUK,21931,12535.759219,1.749475,0.999886,0.187517,0.242843,0.538021,0.013686,0.017383,0.000000,0.430360,0.031069,0.569090,100070105
